## Prerequisites

Runtime: Python 3, T4 GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# SmolVLM2 was added to core transformers in 4.49.0.
# No bitsandbytes needed: 2B params × 2 bytes = ~4 GB fp16, well within T4's 15 GB.
# No trust_remote_code, no flash_attn dependency.
%pip install -q "transformers>=4.49.0" accelerate Pillow

In [ ]:
from transformers import AutoProcessor, AutoModelForVision2Seq

In [ ]:
import torch

In [ ]:
from pathlib import Path
from PIL import Image

WORKING_DIR = Path('/content/drive/MyDrive/aiOCR')
MODEL_NAME = 'HuggingFaceTB/SmolVLM2-2B-Instruct'

# SmolVLM2-2B: SigLIP vision encoder + SmolLM2-2B decoder.
# ~4 GB fp16 — no quantization needed on T4.
# AutoModelForVision2Seq is the correct class (encoder-injected causal LM).
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
).eval().cuda()

In [ ]:
IMAGE_FILES = [
    WORKING_DIR / 'images/CCundinamarca/CCundinamarca_page_1.png',
    WORKING_DIR / 'images/CCundinamarca/CCundinamarca_page_46.png',
    WORKING_DIR / 'images/CO_18180627/CO_18180627_page_1.png',
    WORKING_DIR / 'images/dmcz_18250101/dmcz_18250101_page_1.png',
    WORKING_DIR / 'images/dmcz_18250101/dmcz_18250101_page_4.png',
    WORKING_DIR / 'images/el-redactor-1/el-redactor-1_page_1.png',
    WORKING_DIR / 'images/el-redactor-1/el-redactor-1_page_2.png',
    WORKING_DIR / 'images/pineda1/pineda1_page_1.png',
    WORKING_DIR / 'images/pineda1/pineda1_page_3.png',
    WORKING_DIR / 'images/pineda1/pineda1_page_4.png',
]

## Inference

In [ ]:
import time

transcription_out = WORKING_DIR / 'transcriptions/SmolVLM2-2B'
transcription_out.mkdir(parents=True, exist_ok=True)

prompt = (
    'Convert the document to plain text, as close to the original as possible '
    '(including typos, print errors, and original grammar and spelling). '
    'Do not add any formatting, markdown, or annotations.'
)

for IMAGE_FILE in IMAGE_FILES:
    image_stem = IMAGE_FILE.stem
    out_path = transcription_out / f'{image_stem}.md'

    if out_path.exists():
        print(f'Skipping (already done): {image_stem}')
        continue

    if not IMAGE_FILE.exists():
        print(f'Skipping (image not found): {IMAGE_FILE}')
        continue

    image = Image.open(IMAGE_FILE).convert('RGB')

    messages = [
        {
            'role': 'user',
            'content': [
                {'type': 'image'},
                {'type': 'text', 'text': prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(
        text=text, images=[image], return_tensors='pt'
    ).to(model.device)
    input_len = inputs['input_ids'].shape[-1]

    t0 = time.time()
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=4096,
            do_sample=False,
        )
    elapsed = time.time() - t0

    generated_ids_trimmed = [out[input_len:] for out in generated_ids]
    transcription = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
    )[0]

    out_path.write_text(transcription, encoding='utf-8')
    print(f'Done in {elapsed:.1f}s — saved: transcriptions/SmolVLM2-2B/{image_stem}.md')

### Saving the output